# Python Notebook

This is a simple Python notebook for experimentation and testing. You can add your code, analysis, and visualizations here.

## GitHub PR Library Update Analyzer

This notebook analyzes GitHub Pull Requests to extract library version updates from POM file changes using local Ollama LLM.

### Prerequisites

1. **GitHub Personal Access Token**: 
   - Go to [GitHub Settings > Developer Settings > Personal Access Tokens > Tokens (classic)](https://github.com/settings/tokens)
   - Click "Generate new token (classic)"
   - Give it a descriptive name (e.g., "PR Analysis Tool")
   - Select the following scopes:
     - `repo` (Full control of private repositories) - needed to read PR data
     - Or just `public_repo` if you only need to access public repositories
   - Click "Generate token"
   - **Copy the token immediately** (you won't be able to see it again)
   - Store it securely

2. **Local Ollama**: 
   - Ensure Ollama is installed and running locally
   - Install a model (e.g., `ollama pull llama3` or `ollama pull mistral`)
   - Verify it's running: `ollama list`

### Step 1: Configure GitHub Access and PR Details

**Setting up your GitHub Token (Recommended):**

Instead of hardcoding your token in the notebook, store it in a `.env` file:

1. Create a `.env` file in the project root:
   ```bash
   echo "GITHUB_TOKEN=your_github_token_here" > .env
   ```

2. The `.env` file is already in `.gitignore` and won't be committed to version control.

3. The notebook will automatically load the token from the `.env` file using `python-dotenv`.

**Note:** The `.env` file should be in the same directory as this notebook.

In [ ]:
# Define the PRs to analyze using full GitHub URLs
PR_URLS = [
    "https://github.com/eg-internal/brand-to-eg-profile-sync/pull/596",
    "https://github.com/eg-internal/brand-to-eg-profile-sync/pull/577"
]

print(f"Configured to analyze {len(PR_URLS)} PR(s):")

In [ ]:
import os
import re
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Load GitHub token from environment variable
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError(
        "GITHUB_TOKEN environment variable not found!\n"
        "Please create a .env file in the project root with:\n"
        "  GITHUB_TOKEN=your_token_here\n"
    )

print(f"✓ GitHub token loaded from .env file")

# Parse all PR URLs
pr_pattern = r'https://github\.com/([^/]+)/([^/]+)/pull/(\d+)'
pr_details = []

for pr_url in PR_URLS:
    match = re.match(pr_pattern, pr_url)
    if not match:
        raise ValueError(f"Invalid GitHub PR URL: {pr_url}")
    
    pr_info = {
        'url': pr_url,
        'owner': match.group(1),
        'repo': match.group(2),
        'number': int(match.group(3))
    }
    pr_details.append(pr_info)
    print(f"  - PR #{pr_info['number']}: {pr_info['owner']}/{pr_info['repo']}")

print(f"\n✓ Parsed {len(pr_details)} PR(s) successfully")

### Step 2: Connect to GitHub and Fetch PR Diffs

This cell connects to GitHub using the PyGithub library and retrieves the diffs for all specified PRs.

In [ ]:
from github import Github
import requests

# Initialize GitHub client
g = Github(GITHUB_TOKEN)

# Fetch diffs for all PRs
pr_diffs = []

for pr_info in pr_details:
    print(f"\nProcessing PR #{pr_info['number']}...")
    print("=" * 80)
    
    # Get the repository
    repo = g.get_repo(f"{pr_info['owner']}/{pr_info['repo']}")
    print(f"Connected to repository: {repo.full_name}")
    
    # Get the pull request
    pr = repo.get_pull(pr_info['number'])
    print(f"PR Title: {pr.title}")
    print(f"PR State: {pr.state}")
    print(f"Files Changed: {pr.changed_files}")
    
    # Fetch the diff
    headers = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3.diff"
    }
    diff_url = f"https://api.github.com/repos/{pr_info['owner']}/{pr_info['repo']}/pulls/{pr_info['number']}"
    response = requests.get(diff_url, headers=headers)
    
    if response.status_code == 200:
        pr_diff = response.text
        print(f"✓ Successfully fetched PR diff ({len(pr_diff)} characters)")
        pr_diffs.append({
            'pr_info': pr_info,
            'diff': pr_diff,
            'title': pr.title
        })
    else:
        print(f"✗ Error fetching diff: {response.status_code}")

print(f"\n{'=' * 80}")
print(f"✓ Successfully fetched {len(pr_diffs)} PR diff(s)")

### Step 3: Extract POM Changes and Analyze with Ollama

This cell processes each PR separately, sends individual requests to Ollama, and aggregates all library updates into a single JSON response.

In [ ]:
import ollama
import re
import json

# Aggregate all library updates
all_library_updates = []

for pr_data in pr_diffs:
    pr_info = pr_data['pr_info']
    pr_diff = pr_data['diff']
    pr_title = pr_data['title']
    
    print(f"\nAnalyzing PR #{pr_info['number']}: {pr_title}")
    print("=" * 80)
    
    # Filter diff for POM files only
    pom_diff_sections = []
    current_file = None
    current_diff = []
    
    for line in pr_diff.split('\n'):
        if line.startswith('diff --git'):
            # Save previous file's diff if it was a POM
            if current_file and 'pom.xml' in current_file:
                pom_diff_sections.append({
                    'file': current_file,
                    'diff': '\n'.join(current_diff)
                })
            # Start new file
            current_file = line
            current_diff = [line]
        else:
            current_diff.append(line)
    
    # Don't forget the last file
    if current_file and 'pom.xml' in current_file:
        pom_diff_sections.append({
            'file': current_file,
            'diff': '\n'.join(current_diff)
        })
    
    print(f"Found {len(pom_diff_sections)} POM file(s) with changes")
    
    # Prepare prompt for Ollama
    if pom_diff_sections:
        # Combine all POM diffs
        combined_pom_diff = "\n\n".join([f"File: {section['file']}\n{section['diff']}" 
                                          for section in pom_diff_sections])
        
        prompt = f"""Analyze the following POM file diff(s) and extract all library/dependency version updates.

Return ONLY a valid JSON object with this structure:
{{
  "library_updates": [
    {{
      "library": "groupId:artifactId",
      "from_version": "X.Y.Z",
      "to_version": "A.B.C"
    }}
  ]
}}

If there are no version updates, return:
{{
  "library_updates": []
}}

Do not include any explanation or markdown formatting, only the JSON object.

Here is the diff:

{combined_pom_diff}
"""
        
        print("Sending request to Ollama...")
        
        # Call Ollama
        response = ollama.chat(
            model='llama3',  # Change to your preferred model
            messages=[{
                'role': 'user',
                'content': prompt
            }],
            format='json'  # Request JSON format
        )
        
        ollama_response = response['message']['content']
        print("✓ Ollama analysis complete")
        
        # Parse JSON response and add PR context
        try:
            result = json.loads(ollama_response)
            pr_updates = result.get('library_updates', [])
            
            # Add PR information to each update
            for update in pr_updates:
                update['pr_number'] = pr_info['number']
                update['pr_url'] = pr_info['url']
                update['pr_title'] = pr_title
                all_library_updates.append(update)
            
            print(f"✓ Found {len(pr_updates)} library update(s) in this PR")
            
        except json.JSONDecodeError as e:
            print(f"✗ Error parsing JSON response: {e}")
    else:
        print("No POM files found in this PR diff")

# Create final aggregated JSON
final_result = {
    "library_updates": all_library_updates,
    "total_updates": len(all_library_updates),
    "prs_analyzed": len(pr_diffs)
}

print("\n" + "=" * 80)
print("FINAL AGGREGATED RESULTS")
print("=" * 80)
print(f"\nTotal PRs analyzed: {final_result['prs_analyzed']}")
print(f"Total library updates found: {final_result['total_updates']}")
print("\nFinal JSON Response:")
print(json.dumps(final_result, indent=2))

In [ ]:
# Print raw JSON result
print("Raw JSON Result:")
print("=" * 80)
print(json.dumps(final_result, indent=2))